# TPG366 Live Pressure Monitor

Live monitoring of all 6 TPG366 pressure sensors with web-based plotting (opens in Chrome tab).

**Sensors:** CH1=IF1, CH2=IF2, CH3=Q1, CH4=QMS, CH5=Transfer, CH6=Depo

**Features:**
- 6 individual subplots with shared time axis, independent y-axes (log scale)
- ~1 Hz update rate
- All values logged to file
- Web-based plot opens in browser tab via Bokeh server

## 1. Imports

In [ ]:
import sys
import os
import time
import logging
import threading
from datetime import datetime, timedelta
from pathlib import Path
from collections import deque

from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, DatetimeTickFormatter
from bokeh.server.server import Server
from tornado.ioloop import IOLoop

sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.pfeiffer.tpg366.tpg366 import TPG366

print("Imports done.")

## 2. Configuration

In [ ]:
# TPG366 connection
COM_PORT = "COM25"
DEVICE_ADDRESS = 10

# Monitoring
POLL_INTERVAL_MS = 1000  # 1 Hz update rate
ROLLING_WINDOW_MIN = 10  # minutes of data visible in plot
MAX_POINTS = 3600        # max data points kept in memory (~1 hour at 1 Hz)

# Bokeh server
BOKEH_PORT = 5006

# Sensor labels
SENSOR_LABELS = {
    1: "IF1",
    2: "IF2",
    3: "Q1",
    4: "QMS",
    5: "Transfer",
    6: "Depo",
}

print("Configuration set.")

## 3. Setup Logger

In [ ]:
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"020_tpg366_monitor_{timestamp}.log"

logger = logging.getLogger(f"TPG366_Monitor_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

## 4. Connect TPG366

In [ ]:
gauge = TPG366(
    device_id="tpg366",
    port=COM_PORT,
    device_address=DEVICE_ADDRESS,
    logger=logger,
)
gauge.connect()
print(f"TPG366 connected on {COM_PORT}")

## 5. Verify Connection

In [ ]:
print(f"Serial Number:    {gauge.get_serial_number()}")
print(f"Device Name:      {gauge.get_device_name()}")
print(f"Firmware Version: {gauge.get_firmware_version()}")
print()
pressures = gauge.read_all_pressures()
for ch, val in pressures.items():
    print(f"  CH{ch} ({SENSOR_LABELS[ch]}): {val} hPa")

## 6. Launch Live Plot (opens in browser)

Running this cell starts a Bokeh server and opens the live plot in your default browser.

**Stop monitoring:** Interrupt the kernel (stop button) to shut down the server.

In [ ]:
# Suppress console logging during live monitoring (file logging continues)
console_handler.setLevel(logging.WARNING)


def make_document(doc):
    """Build the Bokeh document with 6 pressure subplots."""

    # Shared data sources for each sensor
    sources = {}
    for ch in range(1, 7):
        sources[ch] = ColumnDataSource(data={"time": [], "pressure": []})

    # Create 6 figures stacked vertically, sharing x-axis
    figures = []
    for ch in range(1, 7):
        label = SENSOR_LABELS[ch]
        p = figure(
            title=f"CH{ch}: {label}",
            x_axis_type="datetime",
            y_axis_type="log",
            height=180,
            width=1000,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        p.line("time", "pressure", source=sources[ch], line_width=2)
        p.yaxis.axis_label = "Pressure [hPa]"
        p.xaxis.formatter = DatetimeTickFormatter(minutes="%H:%M", hours="%H:%M")

        # Share x_range with first figure
        if ch == 1:
            first_fig = p
        else:
            p.x_range = first_fig.x_range

        # Only show x-axis label on bottom plot
        if ch < 6:
            p.xaxis.axis_label = ""
        else:
            p.xaxis.axis_label = "Time"

        figures.append(p)

    layout = column(*figures, sizing_mode="stretch_width")
    doc.add_root(layout)
    doc.title = "TPG366 Pressure Monitor"

    def update():
        """Periodic callback: read pressures, log, and update plot."""
        try:
            pressures = gauge.read_all_pressures()
            now = datetime.now()

            for ch in range(1, 7):
                val = pressures.get(ch)
                if val is not None:
                    # Log using same format as custom_logger
                    logger.info(
                        f"tpg366//{COM_PORT}//Sensor_CH{ch}_Press={val}//hPa"
                    )

                    # Stream new data point (with rollover to limit memory)
                    sources[ch].stream(
                        {"time": [now], "pressure": [val]},
                        rollover=MAX_POINTS,
                    )

            # Update x-range to show rolling window
            x_end = now
            x_start = now - timedelta(minutes=ROLLING_WINDOW_MIN)
            first_fig.x_range.start = x_start
            first_fig.x_range.end = x_end

        except Exception as e:
            logger.error(f"Update failed: {e}")

    doc.add_periodic_callback(update, POLL_INTERVAL_MS)


# Start Bokeh server in a background thread
server = Server(
    {'/': make_document},
    port=BOKEH_PORT,
    allow_websocket_origin=[f"localhost:{BOKEH_PORT}"],
)
server.start()
print(f"Bokeh server running at http://localhost:{BOKEH_PORT}/")
print("Opening browser...")
server.io_loop.add_callback(server.show, "/")

try:
    server.io_loop.start()
except KeyboardInterrupt:
    print("\nStopping server...")
finally:
    server.stop()
    console_handler.setLevel(logging.DEBUG)
    logger.info("Bokeh server stopped.")
    print("Server stopped.")

## 7. Disconnect

In [ ]:
gauge.disconnect()
print("TPG366 disconnected.")